In [ ]:
import time

from pyspark.sql import functions as F

from config import bronze_table

BATCH_SIZE = 10
NUM_BATCHES = 5
INTERVAL_SECONDS = 5

In [ ]:
review_table = bronze_table("review")
sampled_cols = ["business_id", "stars", "useful", "funny", "cool", "text"]
bronze_columns = [
    "review_id", "user_id", "business_id", "stars", "useful", "funny",
    "cool", "text", "date", "_source_file", "_ingested_at",
]

total_reviews = spark.table(review_table).count()
sample_fraction = min(1, (BATCH_SIZE * 3) / total_reviews)

for batch in range(1, NUM_BATCHES + 1):
    simulated_df = (
        spark.table(review_table)
        .select(*sampled_cols)
        .sample(fraction=sample_fraction)
        .limit(BATCH_SIZE)
        .withColumn("review_id", F.expr("uuid()"))
        .withColumn("user_id", F.expr("uuid()"))
        .withColumn("date", F.current_timestamp())
        .withColumn("_source_file", F.lit("streaming_simulator"))
        .withColumn("_ingested_at", F.current_timestamp())
        .select(*bronze_columns)
    )

    simulated_df.write.format("delta").mode("append").saveAsTable(review_table)

    print(f"batch {batch}/{NUM_BATCHES}: {simulated_df.count()} reviews simuladas inseridas em {review_table}")

    if batch < NUM_BATCHES:
        time.sleep(INTERVAL_SECONDS)